###Intro to CNN and Comp. Vision using tensorflow

Get the data

In [ ]:
import zipfile
# images are from the Food101 dataset
!wget https://storage.googleapis.com/ztm_tf_course/food_vision/pizza_steak.zip

# unzip the downloaded file
zip_ref = zipfile.ZipFile("pizza_steak.zip")
zip_ref.extractall()
zip_ref.close()

In [ ]:
# inspect the data

!ls pizza_steak # ls stands for list

In [ ]:
!ls pizza_steak/train/

In [ ]:
# !ls pizza_steak/train/pizza/

In [ ]:
import os

# walk through the dir and list the no. files in pizza_steak directory
for dirpath, dirnames, filenames in os.walk("pizza_steak"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} files in {dirpath}")

In [ ]:
# another way to see the no. of files in a directory
num_steak_images = len(os.listdir("pizza_steak/train/steak"))
num_steak_images # no. of steak images in train directory

In [ ]:
# get classnames programatically

import pathlib
import numpy as np

data_dir = pathlib.Path("pizza_steak/train")
# data_dir
class_names = np.array(sorted(item.name for item in data_dir.glob('*')))
print(class_names)

Get to know the data

In [ ]:
import matplotlib.pyplot as plt # to plot pixels
import matplotlib.image as mpimg # for pixel to array conversion to feed into the model
import random

In [ ]:
def view_random_image(target_dir, target_class):
  # setup target directory
  target_folder = target_dir + target_class
  print(target_folder)

  # get a random image path
  random_image = random.sample(os.listdir(target_folder), 1) # this means to randomly sample 1 image from target_folder
  # random.sample returns a list, use [0] to get the item as a str
  print(f"name: {random_image[0]}")

  # read in the image and plot using matplotlib
  img = mpimg.imread(target_folder + "/" + random_image[0]) # converts image to array
  return(img)
  plt.title(target_class)
  plt.axis("off") # turns of the x, y axis grid

  # get the shape of the image
  print(f"Image shape: {img.shape}") # shape: (height, width, color channels)

# img = view_random_image("pizza_steak/train/", "steak")
view_random_image("pizza_steak/train/", "steak")



Building our model (end to end example)

In [ ]:
# setting seed for reproduceability

import tensorflow as tf
import numpy as np
import random
import os

def set_seeds(seed=42):
    # 1. Set the Python standard library seed
    random.seed(seed)

    # 2. Set the NumPy seed (for data shuffling/arrays)
    np.random.seed(seed)

    # 3. Set the TensorFlow global seed (for weight initialization)
    tf.random.set_seed(seed)

    # 4. Optional: Force TensorFlow to use deterministic operations
    # (Note: This can sometimes slow down training on GPUs)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

    print(f"Random seed set to: {seed}")

# Call the function
set_seeds(42)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:

# preprocess data (get all pixel values between 0, 1)
train_datagen = ImageDataGenerator(rescale = 1./255)
valid_datagen = ImageDataGenerator(rescale = 1./255)

# set path
train_dir = "pizza_steak/train";
test_dir = "pizza_steak/test";

# import data from directories and turn it into batches
train_data = train_datagen.flow_from_directory(train_dir,
                                               target_size = (224, 224),
                                               batch_size = 32,
                                               class_mode = "binary",
                                               );

'''
target_size resizes all images to 224x224 pixels
batch_size is the no. of images that get processed at a time
class_mode = binary tells it there are only 2 optionz
'''

valid_data = valid_datagen.flow_from_directory(test_dir,
                                               batch_size = 32,
                                               target_size = (224, 224),
                                               class_mode = "binary");

# buid a CNN model (same as Tiny VGG on the CNN explainer website)

model_1 = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(filters = 10,
                           kernel_size = 3,
                           activation = "relu",
                           input_shape = (224, 224, 3)),
    tf.keras.layers.Conv2D(10, 3, activation = "relu"),
    tf.keras.layers.MaxPool2D(pool_size = 2,
                              padding = "valid"),
    tf.keras.layers.Conv2D(10, 3, activation = "relu"),
    tf.keras.layers.Conv2D(10, 3, activation = "relu"),
    tf.keras.layers.MaxPool2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation = "sigmoid")
])


# compile our cnn
model_1.compile(loss = "BinaryCrossentropy",
                optimizer = tf.keras.optimizers.Adam(),
                metrics = ["accuracy"])

# fit the model
history_1 = model_1.fit(train_data, epochs = 5,
                        steps_per_epoch = len(train_data),
                        validation_data = valid_data,
                        validation_steps = len(valid_data))

In [ ]:
model_1.summary()

###Understanding the data

In [ ]:
# visualize data
plt.figure()

plt.subplot(1, 2, 1)
steak_image = view_random_image("pizza_steak/train/", "steak"); # returns img array
plt.imshow(steak_image); plt.axis("off")

plt.subplot(1, 2, 2)
pizza_image = view_random_image("pizza_steak/train/", "pizza");
plt.imshow(pizza_image); plt.axis("off")

Turn the data into batches

In [ ]:
# check out which gpu we're on
!nvidia-smi

In [ ]:
# create train and test data generators and rescale the data
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_datagen = ImageDataGenerator(rescale = 1./255)
test_datagen = ImageDataGenerator(rescale = 1/255.)

# This code initializes the "conveyor belt" system for your image data.

In [ ]:
# load in our image data from directories and turn them into batches

train_data = train_datagen.flow_from_directory(directory = train_dir,
                                               target_size = (224, 224),
                                               class_mode = "binary",
                                               batch_size = 32)

test_data = train_datagen.flow_from_directory(directory = test_dir,
                                               target_size = (224, 224),
                                               class_mode = "binary",
                                               batch_size = 32)

In [ ]:
# get a sample of a train data batch
images, labels = next(train_data)
len(images), len(labels) # its 32 due to our batch_size

In [ ]:
# how many batches are there?
len(train_data) # ~1500/32

In [ ]:
# get the first 2 images and their shape
# images[:2], images[0].shape

# this is to ensure the array is normalized

In [ ]:
# view the first batch of labels
labels # give, pizza or steak (training labels)

### Create a CNN model (start with a baseline)



In [ ]:
# make creating of our model a little easier
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPool2D, Activation
from tensorflow.keras import Sequential

In [ ]:
# create our baseline model

model_2 = Sequential([
    Conv2D(filters =10,
           kernel_size = 3,
           strides = 1,
           activation = "relu",
           input_shape = (224, 224, 3)), # imput_shape is needed only in the 1st layer
    Conv2D(10, 3, activation = "relu"), # the other hyperparameters is given via a handshake from the 1st layer
    Conv2D(10,3, activation = "relu"),
    Flatten(),
    Dense(1, activation = "sigmoid") # output layer => 1 neuron
])

In [ ]:
# compile model_2
model_2.compile(loss = "binary_crossentropy",
                optimizer = Adam(),
                metrics = ["accuracy"])

In [ ]:
model_2.summary()

In [ ]:
# check lengths of training and test data generators
len(train_data), len(test_data)

# fit model_2
history_2 = model_2.fit(train_data,
                        epochs = 5,
                        steps_per_epoch = len(train_data),
                        validation_data = test_data,
                        validation_steps = len(test_data))

'''
Because you are now using ImageDataGenerator (specifically flow_from_directory),
the labels are already "baked into" the train_data object itself. So  dont need
to pass train_labels seperately while fitting model_2
''';

In [ ]:
# evaluate model_2
model_2.evaluate(test_data)

Evaluate the baseline model

In [ ]:
# plot the training curves
import pandas as pd
pd.DataFrame(history_2.history).plot()

In [ ]:
# Plot the validation and training data separately
def plot_loss_curves(history):
  """
  Returns separate loss curves for training and validation metrics.
  """
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  accuracy = history.history['accuracy']
  val_accuracy = history.history['val_accuracy']

  epochs = range(len(history.history['loss']))

  # Plot loss
  plt.plot(epochs, loss, label='training_loss')
  plt.plot(epochs, val_loss, label='val_loss')
  plt.title('Loss')
  plt.xlabel('Epochs')
  plt.legend()

  # Plot accuracy
  plt.figure()
  plt.plot(epochs, accuracy, label='training_accuracy')
  plt.plot(epochs, val_accuracy, label='val_accuracy')
  plt.title('Accuracy')
  plt.xlabel('Epochs')
  plt.legend();


  '''
  Ideally training and validation curves are supposed to be similar.
  If training loss is dec but validation moss is inc => overfitting
  '''

In [ ]:
plot_loss_curves(history_2)

In [ ]:
# Create the model (this can be our baseline, a 3 layer Convolutional Neural Network)

model_3 = Sequential([
  Conv2D(10, 3, activation='relu', input_shape=(224, 224, 3)),
  MaxPool2D(pool_size=2), # reduce number of features by half
  Conv2D(10, 3, activation='relu'),
  MaxPool2D(),
  Conv2D(10, 3, activation='relu'),
  MaxPool2D(),
  Flatten(),
  Dense(1, activation='sigmoid')
])


In [ ]:
# copile the model
model_3.compile(loss="BinaryCrossentropy",
                optimizer = Adam(),
                metrics = ["accuracy"])

In [ ]:
# fit the model
history_3 = model_3.fit(train_data,
                        epochs = 5,
                        steps_per_epoch = len(train_data),
                        validation_data = test_data,
                        validation_steps = len(test_data))

In [ ]:
model_3.evaluate(test_data)

In [ ]:
plot_loss_curves(history_3)

In [ ]:
model_3.summary()

Reduce overfitting with augmentation

In [ ]:

# Create ImageDataGenerator training instance with data augmentation
train_datagen_augmented = ImageDataGenerator(rescale=1/255.,
                                             rotation_range=0.2, # how much do you want to rotate an image?
                                             shear_range=0.2, # how much do you want to shear an image?
                                             zoom_range=0.2, # zoom in randomly on an image
                                             width_shift_range=0.2, # move your iamge around on the x-axis
                                             height_shift_range=0.2, # move your image around on the y-axis
                                             horizontal_flip=True) # do you want to flip and image?

# Create ImageDataGenerator without data augmentation
train_datagen = ImageDataGenerator(rescale=1/255.)

# Create ImageDataGenerator without data augmentation for the test dataset
test_datagen = ImageDataGenerator(rescale=1/255.)

In [ ]:
# Import data and augment it from training directory
print("Augmented training data:")
train_data_augmented = train_datagen_augmented.flow_from_directory(train_dir,
                                                                   target_size=(224, 224),
                                                                   batch_size=32,
                                                                   class_mode="binary",
                                                                   shuffle=False) # for demonstration purposes only

# Create non-augmented train data batches
print("Non-augmented training data:")
train_data = train_datagen.flow_from_directory(train_dir,
                                               target_size=(224, 224),
                                               batch_size=32,
                                               class_mode="binary",
                                               shuffle=False)

IMG_SIZE = (224, 224)
# Create non-augmented test data batches
print("Non-augmented test data:")
test_data = test_datagen.flow_from_directory(test_dir,
                                             target_size=IMG_SIZE,
                                             batch_size=32,
                                             class_mode="binary")

In [ ]:
# Get sample data batches
images, labels = next(train_data)
augmented_images, augmented_labels = next(train_data_augmented) # note: labels aren't augmented... only data (images)

In [ ]:
# Show original image and augmented image
import random
random_number = random.randint(0, 32) # our batch sizes are 32...
print(f"showing image number: {random_number}")
plt.imshow(images[random_number])
plt.title(f"Original image")
plt.axis(False)
plt.figure()
plt.imshow(augmented_images[random_number])
plt.title(f"Augmented image")
plt.axis(False);

In [ ]:
model_4 = Sequential([
  Conv2D(10, 3, activation='relu', input_shape=(224, 224, 3)),
  MaxPool2D(pool_size=2),
  Conv2D(10, 3, activation="relu"),
  MaxPool2D(),
  Conv2D(10, 3, activation="relu"),
  MaxPool2D(),
  Flatten(),
  Dense(1, activation="sigmoid")
])


In [ ]:
model_4.compile(loss="binary_crossentropy",
                optimizer=Adam(),
                metrics=["accuracy"])

In [ ]:
# Fit the model
history_4 = model_4.fit(train_data_augmented, # fitting model_4 on augmented training data
                        epochs=5,
                        steps_per_epoch=len(train_data_augmented),
                        validation_data=test_data,
                        validation_steps=len(test_data))

In [ ]:
plot_loss_curves(history_4)

In [ ]:
# Import data and augment it and shuffle from training directory
train_data_augmented_shuffled = train_datagen_augmented.flow_from_directory(train_dir,
                                                                            target_size=(224, 224),
                                                                            class_mode="binary",
                                                                            batch_size=32,
                                                                            shuffle=True) # shuffle data this time


In [ ]:
# Create the model (same as model_5 and model_6)
model_5 = Sequential([
  Conv2D(10, 3, activation="relu", input_shape=(224, 224, 3)),
  MaxPool2D(),
  Conv2D(10, 3, activation="relu"),
  MaxPool2D(),
  Conv2D(10, 3, activation="relu"),
  MaxPool2D(),
  Flatten(),
  Dense(1, activation="sigmoid")
])

# Compile the model
model_5.compile(loss="binary_crossentropy",
                optimizer=Adam(),
                metrics=["accuracy"])

# Fit the model
history_5 = model_5.fit(train_data_augmented_shuffled, # we're fitting on augmented and shuffled data now
                        epochs=5,
                        steps_per_epoch=len(train_data_augmented_shuffled),
                        validation_data=test_data,
                        validation_steps=len(test_data))

In [ ]:
model_5.evaluate(test_data)

In [ ]:
plot_loss_curves(history_5)

In [ ]:
'''
Since we've already beaten our baseline, there are a few things we could try to contine to improve our model:

Increase the number of model layers (e.g. add more Conv2D/MaxPool2D layers)
Increase the number of filters in each convolutional layer (e.g. from 10 to 32 or even 64)
Train for longer (more epochs)
Find an ideal learning rate
Get more data (give the model more opportunities to learn)
Use transfer learning to leverage what another image model has learn and adjust it for our own use case
'''

### Making a prediction with our trained model on our custom data

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Download the raw image file from GitHub
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/images/03-steak.jpeg -O 03-steak.jpeg

# View our example image
steak = mpimg.imread("03-steak.jpeg")
plt.imshow(steak)
plt.axis(False)

In [ ]:
# check shape
steak.shape

In [ ]:
def preprocess_image(image_path, target_size=(224, 224)):
  # Load image
  img = mpimg.imread(image_path)
  # Resize image
  img = tf.image.resize(img, target_size)
  # Rescale pixel values to 0-1 (as done during training)
  img = img / 255.0
  # Add a batch dimension (model expects input in batches)
  img = tf.expand_dims(img, axis=0)
  return img

# Preprocess the custom steak image
preprocessed_steak = preprocess_image("03-steak.jpeg")

# Make a prediction with model_5
prediction_prob = model_5.predict(preprocessed_steak)
prediction_prob

In [ ]:
prediction_class = class_names[round(prediction_prob[0][0].item())]
print(f"The predicted output is: {prediction_class}")
print(f"Confidence of {prediction_prob[0][0].item()*100}%")

### Save and load our model

In [ ]:
# Save model_5
model_5.save("Model_5_trained_pizza_steak_binary.keras")

In [ ]:
# load in a trained model
loaded_model_5 = tf.keras.models.load_model("Model_5_trained_pizza_steak_binary.keras")
loaded_model_5.evaluate(test_data)

In [ ]:
# eval should be same as original since a random seed is set in the starting cells
model_5.evaluate(test_data)